# Partie 1 — Classification de vibrations (Google Colab)
**Classes :** `idle` (0) · `shake` (1)  
**Input :** fenêtres 50 samples × 3 axes = 150 features  
**Export :** TFLite int8 → tableau C pour Arduino

## 0. Installation des dépendances

In [ ]:
!pip install tensorflow numpy pandas scikit-learn matplotlib seaborn -q

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import seaborn as sns
import os, io
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from google.colab import files

os.makedirs('models', exist_ok=True)
print('TensorFlow:', tf.__version__)

## 2. Upload des CSV

In [ ]:
# ⚠️ Une fenêtre va s'ouvrir → sélectionne repos.csv ET vibration.csv
uploaded = files.upload()

## 3. Chargement et préparation des données

In [ ]:
df_idle  = pd.read_csv(io.BytesIO(uploaded['repos.csv']),     sep=';')
df_shake = pd.read_csv(io.BytesIO(uploaded['vibration.csv']), sep=';')

print(f'idle  : {len(df_idle)} lignes  → colonnes : {list(df_idle.columns)}')
print(f'shake : {len(df_shake)} lignes → colonnes : {list(df_shake.columns)}')

# Visualisation rapide
fig, axes = plt.subplots(2, 1, figsize=(12, 6))
df_idle.plot(ax=axes[0], title='idle — aX, aY, aZ')
df_shake.plot(ax=axes[1], title='shake — aX, aY, aZ')
plt.tight_layout()
plt.show()

## 4. Création des fenêtres glissantes

In [ ]:
WINDOW_SIZE = 50
N_FEATURES  = 3
CLASS_NAMES = ['idle', 'shake']

def make_windows(df_class, label):
    data = df_class[['aX','aY','aZ']].values.astype(np.float32)
    X, y = [], []
    for i in range(0, len(data) - WINDOW_SIZE + 1, WINDOW_SIZE):
        X.append(data[i:i+WINDOW_SIZE].flatten())
        y.append(label)
    return np.array(X), np.array(y)

X_idle,  y_idle  = make_windows(df_idle,  0)
X_shake, y_shake = make_windows(df_shake, 1)

X = np.vstack([X_idle, X_shake])
y = np.concatenate([y_idle, y_shake])

print(f'Fenêtres idle  : {len(X_idle)}')
print(f'Fenêtres shake : {len(X_shake)}')
print(f'Shape X        : {X.shape}')  # (N, 150)

## 5. Normalisation

In [ ]:
X_min = X.min(axis=0)
X_max = X.max(axis=0)
X_norm = (X - X_min) / (X_max - X_min + 1e-8)

np.save('models/X_min.npy', X_min)
np.save('models/X_max.npy', X_max)

print(f'Min global : {X.min():.4f}')
print(f'Max global : {X.max():.4f}')
print('→ Ces valeurs seront utilisées dans le sketch Arduino')

## 6. Split train / test

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_norm, y, test_size=0.2, random_state=42, stratify=y
)

y_train_oh = tf.keras.utils.to_categorical(y_train, num_classes=2)
y_test_oh  = tf.keras.utils.to_categorical(y_test,  num_classes=2)

print(f'Train : {X_train.shape} | Test : {X_test.shape}')

## 7. Modèle

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Dense(64, activation='relu', input_shape=(WINDOW_SIZE * N_FEATURES,)),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(2,  activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

## 8. Entraînement

In [ ]:
history = model.fit(
    X_train, y_train_oh,
    epochs=50,
    batch_size=16,
    validation_split=0.2,
    callbacks=[tf.keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)],
    verbose=1
)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history.history['accuracy'],     label='Train')
ax1.plot(history.history['val_accuracy'], label='Validation')
ax1.set_title('Accuracy'); ax1.legend()
ax2.plot(history.history['loss'],     label='Train')
ax2.plot(history.history['val_loss'], label='Validation')
ax2.set_title('Loss'); ax2.legend()
plt.tight_layout()
plt.savefig('models/training_curves.png')
plt.show()

## 9. Évaluation

In [ ]:
loss, acc = model.evaluate(X_test, y_test_oh, verbose=0)
print(f'\n✅ Test Accuracy : {acc*100:.1f}%\n')

y_pred = np.argmax(model.predict(X_test), axis=1)
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title('Confusion Matrix')
plt.ylabel('Réel'); plt.xlabel('Prédit')
plt.tight_layout()
plt.savefig('models/confusion_matrix.png')
plt.show()

## 10. Export TFLite (quantized int8)

In [ ]:
def representative_dataset():
    for i in range(min(100, len(X_train))):
        yield [X_train[i:i+1]]

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type  = tf.int8
converter.inference_output_type = tf.int8

tflite_model = converter.convert()

with open('models/vibrations_model.tflite', 'wb') as f:
    f.write(tflite_model)

print(f'✅ Modèle TFLite : {len(tflite_model)/1024:.1f} KB')

## 11. Génération du tableau C pour Arduino

In [ ]:
with open('models/vibrations_model.tflite', 'rb') as f:
    model_bytes = f.read()

c_array = ', '.join([f'0x{b:02x}' for b in model_bytes])

# Affiche les valeurs de normalisation à reporter dans le sketch Arduino
global_min = float(X.min())
global_max = float(X.max())
print(f'⚠️  Dans inference_vibrations.ino, mets :')
print(f'   const float GLOBAL_MIN = {global_min:.4f}f;')
print(f'   const float GLOBAL_MAX = {global_max:.4f}f;')

c_header = f'''// Modèle TFLite — Classification vibrations
// Classes : 0=idle, 1=shake
// Taille  : {len(model_bytes)} bytes
// GLOBAL_MIN = {global_min:.4f}f
// GLOBAL_MAX = {global_max:.4f}f

alignas(8) const unsigned char vibrations_model[] = {{
  {c_array}
}};
const unsigned int vibrations_model_len = {len(model_bytes)};
'''

with open('models/vibrations_model.h', 'w') as f:
    f.write(c_header)

print('\n✅ vibrations_model.h généré !')

## 12. Téléchargement du fichier .h

In [ ]:
# Télécharge le header C → à placer dans le dossier du sketch inference_vibrations.ino
files.download('models/vibrations_model.h')
files.download('models/training_curves.png')
files.download('models/confusion_matrix.png')
print('✅ Fichiers téléchargés !')